# nlu-ablations: train and evaluate on Google Colab

This notebook trains the models of the two studies on a Colab GPU and evaluates them.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Every group of experiments can be trained on its own: run only the cells you need, or change the names after
   `--exp`.
3. After every group, the checkpoints and the results (JSON records, loss curves and logs) are copied to Google
   Drive, so a disconnection does not lose the groups that are already finished.

On a T4 the whole run takes about 2 hours and 20 minutes.

In [ ]:
import torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU")

## Setup

In [ ]:
import os

REPO = "https://github.com/giampyhatesyou/nlu-ablations.git"
BRANCH = "main"  # branch with the code to run
ROOT = "/content/nlu-ablations"

if not os.path.isdir(ROOT):
    !git clone -q --depth 1 --branch {BRANCH} {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
%pip install -q -e {ROOT}

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/nlu-ablations"  # where checkpoints and results are copied
LOGS = f"{ROOT}/results/logs"
os.makedirs(LOGS, exist_ok=True)


def save_to_drive():
    """Copy checkpoints/ and results/ (JSON records, loss curves, logs) to Google Drive."""
    if not os.path.isdir("/content/drive/MyDrive"):
        print("Google Drive is not mounted: the results stay only in this runtime")
        return
    for sub in ("checkpoints", "results"):
        src = os.path.join(ROOT, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(DRIVE_DIR, sub), dirs_exist_ok=True)
    print("checkpoints and results copied to", DRIVE_DIR)

## Language modelling, part A: LSTM, dropout, AdamW

In [ ]:
!python -u -m nlu_ablations.lm --train --exp lstm lstm_dropout lstm_dropout_adamw 2>&1 | tee {LOGS}/lm_part_a.log
save_to_drive()

## Language modelling, part B: weight tying, variational dropout, NT-AvSGD

The longest group: after the trigger, NT-AvSGD keeps training while the averaged weights improve on dev.

In [ ]:
!python -u -m nlu_ablations.lm --train --exp wt wt_vd wt_vd_ntavsgd 2>&1 | tee {LOGS}/lm_part_b.log
save_to_drive()

## Intent classification and slot filling, part A: LSTM, BiLSTM, BiLSTM + dropout (5 runs each)

In [ ]:
!python -u -m nlu_ablations.slu --train --exp lstm bilstm bilstm_dropout 2>&1 | tee {LOGS}/slu_part_a.log
save_to_drive()

## Intent classification and slot filling, part B: BERT (5 runs)

The checkpoint of the best run is about 420 MB.

In [ ]:
!python -u -m nlu_ablations.slu --train --exp bert 2>&1 | tee {LOGS}/slu_part_b.log
save_to_drive()

## Tables of the papers

Computed from the JSON records in `results/` only: the tables of the README, plus the LaTeX tables and the figure
of the papers in `reports/`.

In [ ]:
!cd {ROOT} && python reports/analysis.py

## Restore the checkpoints from Drive (optional)

In a new runtime: run the setup cells, then this one, to evaluate the saved models without training them again.

In [ ]:
for sub in ("checkpoints", "results"):
    src = os.path.join(DRIVE_DIR, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(ROOT, sub), dirs_exist_ok=True)
        print("restored", sub)

## Evaluate the checkpoints on the test sets

Without `--train`, both commands load the checkpoints in `checkpoints/` and evaluate them.

In [ ]:
!python -m nlu_ablations.lm
!python -m nlu_ablations.slu

## Download the results

Checkpoints and results in one zip file; the BERT checkpoint alone is about 420 MB. Unzipped in the root of the
repository, the two folders replace `checkpoints/` and `results/`.

In [ ]:
OUT = "/content/nlu-ablations-results"
shutil.rmtree(OUT, ignore_errors=True)
for sub in ("checkpoints", "results"):
    src = os.path.join(ROOT, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(OUT, sub))
archive = shutil.make_archive(OUT, "zip", OUT)

from google.colab import files
files.download(archive)